# Local Recent Blocks Power Sampling Ablation

This notebook tests the new pivot: **local power sampling** can be much faster than global power sampling while matching performance.

The key ablation is `LOCAL_RECENT_BLOCKS`, which controls how much of the recent generated trajectory an MH move may resample.

For power sampling with `MAX_NEW_TOKENS = 1024` and `BLOCK_NUM = 16`:

```text
jump_size = MAX_NEW_TOKENS // BLOCK_NUM = 64
LOCAL_RECENT_BLOCKS = 1  -> latest 64 tokens
LOCAL_RECENT_BLOCKS = 2  -> latest 128 tokens
LOCAL_RECENT_BLOCKS = 4  -> latest 256 tokens
LOCAL_RECENT_BLOCKS = 16 -> latest 1024 tokens, equivalent to global for this run
```


## Setup

Run this on a GPU runtime. The notebook expects `power_sampling_common.py` to be in the same directory. That module was extracted from the Qwen3 notebook and contains prompt formatting, MATH answer parsing, model loading, and the original sampler utilities.


In [ ]:
# Install dependencies when running in a fresh Colab/runtime.
%pip install -q transformers accelerate pandas tqdm torch


In [ ]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# If running in Colab, upload/copy power_sampling_common.py next to this notebook.
sys.path.append(str(Path.cwd()))

from power_sampling_common import (
    MODEL_REPOS,
    AutoregressiveSampler,
    answers_match,
    encode_text_prompt,
    ensure_math500,
    format_prompt,
    infer_model_device,
    load_generation_model,
    load_math500,
    load_text_processor,
    naive_temp,
    normalize_math_answer,
    parse_answer,
    print_cuda_memory,
    select_shard,
    set_seed,
    sync_cuda,
)


## Configuration

Keep the first run small. For a fuller ablation, increase `MAX_PROBLEMS` after confirming output CSVs look sane.


In [ ]:
MODEL_KEY = "qwen3_8b"
BATCH_IDX = 0
SEED = 0
TEMPERATURE = 0.25
MCMC_STEPS = 4
MAX_NEW_TOKENS = 1024
BLOCK_NUM = 16
LOCAL_RECENT_BLOCKS_GRID = [1, 2, 4, 8, 16]
MAX_PROBLEMS = 10
USE_COT = True
DATA_PATH = "MATH500.json"
SAVE_DIR = "results/local_recent_blocks"

assert MAX_NEW_TOKENS % BLOCK_NUM == 0
JUMP_SIZE = MAX_NEW_TOKENS // BLOCK_NUM
print("jump_size:", JUMP_SIZE)
print("recent-token windows:", {n: n * JUMP_SIZE for n in LOCAL_RECENT_BLOCKS_GRID})


## Local Recent-Blocks Sampler

This is the only algorithmic change from local power sampling. The old local version used the most recent one block:

```python
lo = max(c, t - jump_size)
```

The generalized version uses the most recent `n` blocks:

```python
lo = max(c, t - local_recent_blocks * jump_size)
```


In [ ]:
def mcmc_power_samp_recent_blocks(
    p,
    context,
    temp,
    mcmc_steps,
    max_new_tokens,
    block_num=16,
    local_recent_blocks=1,
):
    c = len(context)
    gen = context.copy() if context is not None else []
    log_probs_norm = []
    log_probs_unnorm = []
    assert max_new_tokens % block_num == 0
    jump_size = int(max_new_tokens // block_num)
    recent_window = int(local_recent_blocks * jump_size)
    print(
        "alpha:", 1 / temp,
        "max_new_tokens:", max_new_tokens,
        "jump_size:", jump_size,
        "local_recent_blocks:", local_recent_blocks,
        "recent_window:", recent_window,
    )

    attempts = 0
    acceptances = 0
    for _ in tqdm(range(block_num), desc=f"blocks recent={local_recent_blocks}"):
        gen, lp_norm, lp_unnorm = naive_temp(p, gen, temp=temp, seq_len=jump_size + len(gen))
        log_probs_norm.extend(lp_norm)
        log_probs_unnorm.extend(lp_unnorm)

        for _ in tqdm(range(mcmc_steps), leave=False):
            attempts += 1
            t = len(gen)
            lo = max(c, t - recent_window)
            idx = np.random.randint(lo, t)
            prop, log_prob_prop, target_log_prob_prop = naive_temp(
                p, gen[:idx], temp=temp, seq_len=t
            )
            s = len(prop)
            assert len(log_prob_prop) == s - idx
            assert len(target_log_prob_prop) == s - idx

            log_prob_cur = log_probs_norm.copy()[idx - c : s - c]
            target_log_prob_cur = log_probs_unnorm.copy()[idx - c : s - c]
            log_r = (
                sum(target_log_prob_prop)
                + sum(log_prob_cur)
                - sum(target_log_prob_cur)
                - sum(log_prob_prop)
            )
            if np.random.rand() < np.exp(log_r):
                acceptances += 1
                gen = prop.copy()
                log_probs_norm[idx - c :] = log_prob_prop.copy()
                log_probs_unnorm[idx - c :] = target_log_prob_prop.copy()

        if p.tokenizer.eos_token_id in gen[c:]:
            eos_idx = c + gen[c:].index(p.tokenizer.eos_token_id)
            gen = gen[: eos_idx + 1]
            log_probs_norm = log_probs_norm[: eos_idx - c + 1]
            log_probs_unnorm = log_probs_unnorm[: eos_idx - c + 1]
            return gen, log_probs_norm, log_probs_unnorm, acceptances / max(1, attempts)

    return gen, log_probs_norm, log_probs_unnorm, acceptances / max(1, attempts)


## Load Dataset And Model


In [ ]:
set_seed(SEED)
math_path = ensure_math500(DATA_PATH)
dataset = load_math500(math_path)
start, end, shard = select_shard(dataset, BATCH_IDX, MAX_PROBLEMS)
print(f"shard [{start}, {end}) of MATH500")

model_str = MODEL_REPOS[MODEL_KEY]
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "model:", model_str)

processor, tokenizer = load_text_processor(model_str)
model = load_generation_model(model_str)
device = infer_model_device(model)
sampler = AutoregressiveSampler(model, tokenizer, device)
print_cuda_memory("after model load")
print("model loaded on", device)


## Run One Ablation Setting

This helper returns one row per MATH problem for a fixed `local_recent_blocks` value.


In [ ]:
def run_recent_blocks_setting(local_recent_blocks):
    rows = []
    for data in tqdm(shard, desc=f"MATH recent_blocks={local_recent_blocks}"):
        question = data["prompt"]
        answer = data["answer"]
        input_text = format_prompt(question, MODEL_KEY, tokenizer, USE_COT)
        model_inputs = encode_text_prompt(processor, tokenizer, input_text, device)
        input_ids = model_inputs["input_ids"]
        prompt_len = input_ids.shape[1]
        prefix = [idx.item() for idx in input_ids[0]]

        sync_cuda(device)
        t0 = time.perf_counter()
        generated, _, _, acceptance_ratio = mcmc_power_samp_recent_blocks(
            sampler,
            prefix,
            TEMPERATURE,
            MCMC_STEPS,
            max_new_tokens=MAX_NEW_TOKENS,
            block_num=BLOCK_NUM,
            local_recent_blocks=local_recent_blocks,
        )
        sync_cuda(device)
        seconds = time.perf_counter() - t0

        generated_ids = generated[prompt_len:]
        completion = tokenizer.decode(generated_ids, skip_special_tokens=True)
        predicted_answer = parse_answer(completion)
        rows.append({
            "question": question,
            "correct_answer": answer,
            "correct_answer_normalized": normalize_math_answer(answer),
            "completion": completion,
            "answer": predicted_answer,
            "answer_normalized": normalize_math_answer(predicted_answer),
            "correct": answers_match(predicted_answer, answer),
            "tokens": len(generated_ids),
            "seconds": seconds,
            "tokens_per_second": len(generated_ids) / max(seconds, 1e-9),
            "acceptance_ratio": acceptance_ratio,
            "local_recent_blocks": local_recent_blocks,
            "recent_window_tokens": local_recent_blocks * JUMP_SIZE,
            "block_num": BLOCK_NUM,
            "jump_size": JUMP_SIZE,
            "mcmc_steps": MCMC_STEPS,
            "temperature": TEMPERATURE,
            "max_new_tokens": MAX_NEW_TOKENS,
        })
        print(
            "recent_blocks:", local_recent_blocks,
            "acceptance_ratio:", acceptance_ratio,
            "correct:", answers_match(predicted_answer, answer),
            "tokens/sec:", len(generated_ids) / max(seconds, 1e-9),
        )
    return pd.DataFrame(rows)


## Run Grid

This writes one detail CSV per `LOCAL_RECENT_BLOCKS` value and one summary CSV for the whole grid.


In [ ]:
out_dir = Path(SAVE_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

all_details = []
summary_rows = []
for local_recent_blocks in LOCAL_RECENT_BLOCKS_GRID:
    df = run_recent_blocks_setting(local_recent_blocks)
    detail_path = out_dir / (
        f"local_recent_blocks_{local_recent_blocks}_"
        f"model{MODEL_KEY}_tokens{MAX_NEW_TOKENS}_blocks{BLOCK_NUM}_"
        f"steps{MCMC_STEPS}_batch{BATCH_IDX}_seed{SEED}.csv"
    )
    df.to_csv(detail_path, index=False)
    print("saved detail:", detail_path)

    all_details.append(df)
    summary_rows.append({
        "local_recent_blocks": local_recent_blocks,
        "recent_window_tokens": local_recent_blocks * JUMP_SIZE,
        "correct": df["correct"].mean(),
        "acceptance_ratio": df["acceptance_ratio"].mean(),
        "tokens_per_second": df["tokens_per_second"].mean(),
        "seconds": df["seconds"].mean(),
        "tokens": df["tokens"].mean(),
        "parsed_answer_rate": df["answer"].notna().mean(),
        "n": len(df),
    })

summary = pd.DataFrame(summary_rows)
summary_path = out_dir / (
    f"summary_model{MODEL_KEY}_tokens{MAX_NEW_TOKENS}_blocks{BLOCK_NUM}_"
    f"steps{MCMC_STEPS}_batch{BATCH_IDX}_seed{SEED}.csv"
)
summary.to_csv(summary_path, index=False)
print("saved summary:", summary_path)
display(summary)


## Quick Analysis

Use this cell after the grid finishes. The expected pattern is a speed/accuracy tradeoff as `local_recent_blocks` grows. `LOCAL_RECENT_BLOCKS = BLOCK_NUM` approximates global power sampling for this fixed-token run.


In [ ]:
summary.sort_values("local_recent_blocks")


In [ ]:
if all_details:
    details = pd.concat(all_details, ignore_index=True)
    display(details.groupby("local_recent_blocks")[["correct", "acceptance_ratio", "tokens_per_second", "seconds", "tokens"]].mean())
    failures = details[~details["correct"]][[
        "local_recent_blocks", "question", "correct_answer_normalized", "answer_normalized", "tokens", "seconds"
    ]]
    display(failures)


## Suggested Next Runs

If the 10-example run is stable, scale in this order:

```python
MAX_PROBLEMS = 50
LOCAL_RECENT_BLOCKS_GRID = [1, 2, 4, 8, 16]
```

Then test sensitivity to block granularity:

```python
BLOCK_NUM = 8   # jump_size = 128
BLOCK_NUM = 32  # jump_size = 32
```

The core research plot should show accuracy versus wall-clock or tokens/sec as a function of recent-window size.
